# Asteroid Rotation Period Pipeline
**LSST/Rubin First Look Commissioning Data**

Three-tier pipeline: Fast screening → Period refinement → Disambiguation

---
**Before running:** clone the repo and set your GitHub token below.

## 0. GitHub setup — run once per Colab session

In [ ]:
# ── GitHub credentials ────────────────────────────────────────────────────────
# Store your token in Colab Secrets (key icon in sidebar) as GITHUB_TOKEN
# Never hardcode tokens in notebooks!

from google.colab import userdata
import os

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USER  = 'YOUR_GITHUB_USERNAME'   # <-- change this
REPO_NAME    = 'asteroid-pipeline'      # <-- change this

# Clone repo (first time) or pull latest (subsequent sessions)
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'

if not os.path.exists(f'/content/{REPO_NAME}'):
    !git clone {REPO_URL} /content/{REPO_NAME}
    print('Repository cloned')
else:
    %cd /content/{REPO_NAME}
    !git pull
    print('Repository updated')

%cd /content/{REPO_NAME}

# Configure git identity for commits
!git config user.email 'your@email.com'   # <-- change this
!git config user.name  'Your Name'         # <-- change this

# ── Results directory — points to Drive so files survive disconnects ────────
# Adjust DRIVE_RESULTS_FOLDER to any folder name you prefer in your Drive.
DRIVE_RESULTS_FOLDER = f"asteroid-pipeline-results"
RESULTS_DIR = f"/content/drive/MyDrive/{DRIVE_RESULTS_FOLDER}"

# Mount Drive if not already mounted
import os
if not os.path.exists('/content/drive/MyDrive'):
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to Drive: {RESULTS_DIR}')


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -r requirements.txt -q
print('Dependencies installed')

In [ ]:
# ── Add src to Python path ────────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/asteroid-pipeline/src')
print('src/ on Python path')

## 1. Configuration

In [ ]:
from config import PipelineConfig, DataConfig, PeriodConfig, TierConfig, OutputConfig

config = PipelineConfig(
    data=DataConfig(
        bq_project    = 'lsst-484623',
        bq_table_full = 'lsst-484623.atlast_photometry.public_obs_x05',
        bands_use     = ['g', 'r', 'i'],
        rmsmag_max    = 0.21,
        min_obs_total = 20,
    ),
    period=PeriodConfig(
        period_min_hr  = 0.5,
        period_max_hr  = 24.0,
        n_grid_coarse  = 8_000,
        n_grid_fine    = 15_000,
        mhaov_nh       = 2,
    ),
    tier=TierConfig(
        snr_threshold      = 3.0,
        min_obs            = 20,
        agreement_tol      = 0.05,
        mhaov_pval_thresh  = 0.001,
        bayesian_ci_thresh = 0.5,
        clean_peak_ratio   = 3.0,
    ),
    output=OutputConfig(
        results_dir  = RESULTS_DIR,
        catalog_file = f'{RESULTS_DIR}/period_catalog.csv',
        log_file     = f'{RESULTS_DIR}/pipeline.log',
        verbose      = True,
    )
)
print('Config ready')


## 2. Load data — BigQuery or CSV

In [ ]:
# ── Authenticate with Google Cloud ────────────────────────────────────────────
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

In [ ]:
from ingestion import load_from_bigquery, load_from_csv, list_objects

# Option A: Load from BigQuery (full dataset)
# df = load_from_bigquery(config=config)

# Option B: Load specific asteroids from BigQuery
# df = load_from_bigquery(provids=['2025 MG21', '2025 MV46'], config=config)

# Option C: Load from local CSV (e.g. uploaded to Colab)
# from google.colab import files
# uploaded = files.upload()  # upload rubin_x05_20250226.csv
# df = load_from_csv('rubin_x05_20250226.csv', config)

# Option D: Load a test sample (quick development run)
df = load_from_bigquery(
    config=config,
    limit=50_000,           # first 50k rows for testing
    date_start='2025-04-01'
)

# Summary
summary = list_objects(df)
print(f'Loaded {len(df):,} observations across {len(summary):,} objects')
summary.head(20)

## 3. Inspect a single asteroid (development / debugging)

In [ ]:
from ingestion import load_single_object
from preprocessing import preprocess
from tier1 import run_tier1
from tier2 import run_tier2
from tier3 import run_tier3
from pipeline import run_single_asteroid

# Pick an asteroid to inspect
PROVID = '2025 MG21'  # <-- change to any provid in your dataset

df_obj = load_single_object(PROVID, df)
data   = preprocess(df_obj, config)

print(f'Asteroid: {data.provid}')
print(f'Observations: {data.n_obs}')
print(f'Bands: {data.band_counts}')
print(f'Baseline: {data.baseline_hr:.1f} hours ({data.baseline_hr/24:.1f} days)')
print(f'Amplitude: {data.amplitude:.3f} mag')
print(f'SNR: {data.snr:.1f}')

In [ ]:
# Run individual tiers and inspect results
t1 = run_tier1(data, config)
print(f'Tier 1: passes={t1.passes}  reason={t1.reject_reason}')
print(f'        GLS best={t1.best_period_gls:.3f}hr  power={t1.gls_power_max:.3f}')

if t1.passes:
    t2 = run_tier2(data, t1, config)
    print(f'Tier 2: passes={t2.passes}  to_tier3={t2.to_tier3}')
    print(f'        MHAOV={t2.best_period_mhaov:.3f}hr  MBLS={t2.best_period_mbls:.3f}hr  CE={t2.best_period_ce:.3f}hr')
    print(f'        F={t2.F_stat:.2f}  p={t2.p_value:.2e}  agreement={t2.agreement}  spread={t2.period_spread_pct:.3f}')

    if t2.to_tier3:
        t3 = run_tier3(data, t2, config)
        print(f'Tier 3: publish_tentative={t3.publish_tentative}  needs_followup={t3.needs_followup}')
        print(f'        Bayes MAP={t3.best_period_bayes:.3f}hr  95%CI=[{t3.ci_lo:.3f},{t3.ci_hi:.3f}]hr')
        print(f'        CLEAN={t3.best_period_clean:.3f}hr  peak_ratio={t3.clean_peak_ratio:.1f}')

## 4. Quick diagnostic plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_asteroid(data, t1=None, t2=None, t3=None):
    """Quick diagnostic plot for one asteroid."""
    band_colors = {'g': '#3B6D11', 'r': '#993C1D', 'i': '#185FA5'}
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'{data.provid}  —  N={data.n_obs}  SNR={data.snr:.1f}  amp={data.amplitude:.3f}mag',
                 fontsize=12, fontweight='bold')

    # Panel 1: detrended lightcurve
    ax = axes[0]
    for band, col in band_colors.items():
        m = data.bands == band
        if m.sum() > 0:
            ax.errorbar(data.t_hrs[m]/24, -data.y_dt[m], yerr=data.dy[m],
                        fmt='o', color=col, ms=3, elinewidth=0.5, alpha=0.7, label=band)
    ax.invert_yaxis()
    ax.set_xlabel('Time (days)'); ax.set_ylabel('Brightness (mag)')
    ax.set_title('Detrended lightcurve'); ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    # Panel 2: periodogram (if available)
    ax = axes[1]
    if t1 is not None and len(t1.gls_power) > 0:
        ax.plot(t1.test_periods, t1.gls_power, color='#185FA5', lw=0.8, label='GLS')
    if t2 is not None and len(t2.mhaov_power) > 0:
        norm = t2.mhaov_power / t2.mhaov_power.max()
        ax.plot(t2.test_periods, norm, color='#D85A30', lw=0.8, label='MHAOV NH=2 (norm)')
        ax.axvline(t2.best_period_mhaov, color='#D85A30', lw=1.5, ls='--')
    ax.set_xlabel('Period (hours)'); ax.set_ylabel('Power')
    ax.set_title('Periodogram'); ax.legend(fontsize=8)
    ax.set_xlim(0.5, min(24, data.baseline_hr))
    ax.grid(True, alpha=0.2)

    # Panel 3: phase fold at best period
    ax = axes[2]
    best_P = None
    if t2 is not None and t2.passes:    best_P = t2.consensus_period
    elif t2 is not None:                best_P = t2.best_period_mhaov
    elif t1 is not None:                best_P = t1.best_period_gls

    if best_P is not None and not np.isnan(best_P):
        for band, col in band_colors.items():
            m = data.bands == band
            if m.sum() > 0:
                ph = (data.t_hrs[m] % best_P) / best_P
                ax.scatter(ph, -data.y_dt[m], color=col, s=8, alpha=0.7, label=band)
        ax.invert_yaxis()
        ax.set_xlabel('Phase'); ax.set_ylabel('Brightness (mag)')
        ax.set_title(f'Phase fold  P={best_P:.3f}hr')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

plot_asteroid(data, t1=t1, t2=t2 if t1.passes else None)

## 5. Run the full pipeline

In [ ]:
from pipeline import run_pipeline

# Run on all objects in the loaded dataset
# For a full run on 92k objects, this will take several hours.
# For a test run, pass a subset:

test_provids = summary[summary['n_obs'] >= 30]['provid'].head(50).tolist()

catalog = run_pipeline(
    df          = df,
    config      = config,
    provids     = test_provids,   # remove to run all
    save_every_n = 10,
)

print(f'\nCatalog: {len(catalog)} rows')
catalog[['provid','reliability','final_period_hr','t2_F_stat','t2_p_value',
         't2_amplitude_mag','t1_snr']].dropna(subset=['final_period_hr']).head(20)

## 6. Commit results to GitHub

In [ ]:
# Stage, commit, and push results
import subprocess
from datetime import datetime

timestamp = datetime.now().strftime('%Y-%m-%d %H:%M')

!git add results/period_catalog.csv results/pipeline.log
!git commit -m "Pipeline run {timestamp}: {len(catalog)} asteroids processed"
!git push origin main

print('Results committed and pushed to GitHub')

## 7. Run tests
Run this cell to verify all modules work correctly after any code changes.

In [ ]:
!cd /content/asteroid-pipeline && python -m pytest tests/test_pipeline.py -v --tb=short